### Structured Output

**Structured outputs from LLMs refer to the practice of constraining a language model's response to follow a predefined, machine-readable format like JSON or an XML schema.** 

* Why Use Structured Outputs
    - Reliability: Guarantees that responses adhere strictly to data types, eliminating missing keys or malformed syntax.
    - No Parsing Needed: Sends data directly into downstream software, databases, or APIs without fragile regex.
    - Fewer Hallucinations: Constrains choices to lower invalid token generation and errors.

### Pydantic
===========================

**Pydantic is a popular Python library used for data validation and setting up data rules using standard type hints**.

* **Key Features**:
    - **Runtime Validation**: Checks if incoming data matches expected types when you create an object, raising errors if data is wrong.
    - **Speed**: Core validation runs in Rust, making it one of the fastest validation tools for Python.
    - **Type Coercion**: Automatically converts compatible data types by default (like changing a string "123" into an integer 123).
    - **JSON Schema**: Easily turns models into JSON schemas for web APIs and external tools

In [1]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

## This function will load all the variable from .env file and will make them available in
## os.environ directory (env_variablea)
load_dotenv()
import os 
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')



if os.environ.get("OPENAI_API_KEY"):
    print("✅ OPENAI_API_KEY Exists.")
else:
    raise ValueError("❌ OPENAI_API_KEY Not Found...")


✅ OPENAI_API_KEY Exists.


In [2]:
llm_openai = ChatOpenAI(model="gpt-5-mini",
                    temperature=0)

In [3]:
from pydantic import BaseModel, Field

class LLMSchema(BaseModel):
    setup: str = Field(description="The setup of the joke")
    punchline: str = Field(description="The punchline of the joke")
    
obj = LLMSchema(**{"setup": "some_setup",
                   "punchline": "some_punchline"})

obj

LLMSchema(setup='some_setup', punchline='some_punchline')

In [4]:
# obj = LLMSchema(**{"ketchup": "some_setup",   ## will throw an error
#                    "punchline": 234})

# obj

In [5]:
llm_structured_output = llm_openai.with_structured_output(LLMSchema)
result = llm_structured_output.invoke("Tell me a joke")

result.punchline

'Because he was outstanding in his field.'

#### Groq model

In [6]:
from langchain_groq import ChatGroq

model = ChatGroq(model="openai/gpt-oss-20b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.17'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x10f042120>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10f042e40>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [7]:
class Movie(BaseModel):
    title: str = Field(description="The title of a Movie")
    year: int = Field(description="This year movie has release")
    director: str = Field(description="Movie is directed by him")
    rating: float = Field(description="Rating of movie between 0-10")

In [8]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.17'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x10f042120>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10f042e40>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of a Movie'

In [9]:
model_with_structure.invoke("Provide details about Movie Swadesh")

Movie(title='Swades', year=2004, director='Ashutosh Gowariker', rating=8.2)

In [12]:
model_with_structure.invoke("Provide details about Movie Tamasha")

Movie(title='Tamasha', year=2015, director='Imtiaz Ali', rating=7.5)

### Message output alongside parsed Structure

In [13]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(...,description="The title of a Movie")
    year: int = Field(...,description="This year movie has release")
    director: str = Field(...,description="Movie is directed by him")
    rating: float = Field(...,description="Rating of movie between 0-10")
    
model_with_structure = model.with_structured_output(Movie,
                                                    include_raw=True)

model_with_structure.invoke("Provide details about Movie Interstellar")

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user: "Provide details about Movie Interstellar". We need to use the function to get details about the movie Interstellar. The function signature: Movie with director, rating, title, year. We should call the function with director "Christopher Nolan" (director of Interstellar), rating 8.6 (approx rating on IMDB?), title "Interstellar", year 2014. We need to call the function.', 'tool_calls': [{'id': 'fc_7458acce-3cbe-4354-b220-1b7aaf89eba9', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.6,"title":"Interstellar","year":2014}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 126, 'prompt_tokens': 157, 'total_tokens': 283, 'completion_time': 0.153700244, 'completion_tokens_details': {'reasoning_tokens': 87}, 'prompt_time': 0.007641781, 'prompt_tokens_details': None, 'queue_time': 0.308710479, 'total_time': 0.161342025}, 'model_name': 'openai/g

### Nested Structure

In [15]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str
    
class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genere: list[str]
    budget: float | None=Field(None, description="Budget in million USD")
    
model_with_structure = model.with_structured_output(MovieDetails)

model_with_structure.invoke("Provide details about Movie Interstellar")

MovieDetails(title='Interstellar', year=2014, cast=[Actor(name='Matthew McConaughey', role='Cooper'), Actor(name='Anne Hathaway', role='Brand'), Actor(name='Jessica Chastain', role='Murph (young)'), Actor(name='Michael Caine', role='Professor Brand'), Actor(name='Ellen Burstyn', role='Murph (adult)')], genere=['Adventure', 'Drama', 'Sci-Fi'], budget=165000000.0)

### **TypeDict**
==========================

**Provides the simpler alternative using Python built-in typing; ideal when we don't need runtime validation.**

In [10]:
from typing import TypedDict

class LLMSchemaTD(TypedDict):
    setup: str = Field(description="The setup of the joke")
    punchline: str = Field(description="The punchline of the joke")
    
obj = LLMSchemaTD(**{"ketchup": "some_setup",      ## won't throw an error because TypedDict is more flexible
                   "punchline": "some_punchline"})

obj['punchline'], obj

('some_punchline', {'ketchup': 'some_setup', 'punchline': 'some_punchline'})

In [11]:
llm_structured_output_typedict = llm_openai.with_structured_output(LLMSchemaTD)

llm_structured_output_typedict.invoke("Tell me a joke")

{'setup': "Why don't scientists trust atoms?",
 'punchline': 'Because they make up everything.'}

In [18]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of a Movie"]
    year: Annotated[int, ...,"This year movie has release"]
    director: Annotated[str, ...,"Movie is directed by him"]
    rating: Annotated[float, ...,"Rating of movie between 0-10"]
    
model_with_typedict = model.with_structured_output(MovieDict)
model_with_typedict.invoke("Provide some detail about Harry Potter")

{'director': 'Chris Columbus',
 'rating': 7.6,
 'title': "Harry Potter and the Philosopher's Stone",
 'year': 2001}

In [19]:
model_with_typedict.invoke("Provide some detail about Avenger")

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [20]:
### Nested structure
class Actor(TypedDict):
    name: str
    role: str
    
class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genere: list[str]
    budget: float | None=Field(None, description="Budget in million USD")
    
model_with_structure = model.with_structured_output(MovieDetails)

model_with_structure.invoke("Provide details about Movie Interstellar")

{'budget': 165000000,
 'cast': [{'name': 'Matthew McConaughey', 'role': 'Joseph Cooper'},
  {'name': 'Anne Hathaway', 'role': 'Amelia Brand'},
  {'name': 'Jessica Chastain', 'role': 'Murph (adult)'},
  {'name': 'Mackenzie Foy', 'role': 'Murph (child)'},
  {'name': 'John Lithgow', 'role': 'Dr. Mann'},
  {'name': 'Michael Caine', 'role': 'Professor Brand'},
  {'name': 'Willem Dafoe', 'role': 'Cooper (older)'}],
 'genere': ['Science Fiction', 'Adventure', 'Drama'],
 'title': 'Interstellar',
 'year': 2014}

In [21]:
model.profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### DataClasses
=======================================

**A DataClass is a class typically containing mainly data, although there ain't really any restriction. and we can create with `@dataclass` decorator.**

In [24]:
from langchain.agents import create_agent
from pydantic import BaseModel, Field

class ContactInfo(BaseModel):
    """Contact Information for a person"""
    name: str = Field(description="Name of a person")
    email: str = Field(description="email-id of a person")
    phone: str = Field(description="phone number of a person")

agent = create_agent(
    model="gpt-5-mini",
    response_format=ContactInfo  
)

result = agent.invoke({
    'messages':[{'role':'user',
                 'content':'Extract contact info from: John Doe, john@test.com, 1-(555)-123-4567'}]
    })

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@test.com, 1-(555)-123-4567', additional_kwargs={}, response_metadata={}, id='5f761f57-7926-4b09-94b0-a227e4fe1d90'),
  AIMessage(content='{"name":"John Doe","email":"john@test.com","phone":"1-(555)-123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 164, 'prompt_tokens': 202, 'total_tokens': 366, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EKRco2jmtObByNNd6iQB3sZcNIsIT', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a06d60-d484-7b71-8e3e-5ed6

In [25]:
print(result["structured_response"])

name='John Doe' email='john@test.com' phone='1-(555)-123-4567'


In [27]:
from langchain.agents import create_agent
from dataclasses import dataclass

@dataclass
class ContactInfo():
    """Contact Information for a person"""
    name: str  ## Name of a person
    email: str ## email-id of a person
    phone: str ## phone number of a person

agent = create_agent(
    model="gpt-5-mini",
    response_format=ContactInfo  
)

result = agent.invoke({
    'messages':[{'role':'user',
                 'content':'Extract contact info from: John Doe, john@test.com, 1-(555)-123-4567'}]
    })

result['structured_response']

ContactInfo(name='John Doe', email='john@test.com', phone='1-(555)-123-4567')